In [2]:
from pathlib import Path
import tensorflow as tf
import pandas as pd
import numpy as np

# ----------------------------------
# PATHS
# ----------------------------------

INPUT_DIR = Path(
    "/Users/erinekman/cb1020/Hud-cancer/data"
)

OUTPUT_DIR = Path(
    "/Users/erinekman/cb1020/Hud-cancer/preprocessed_data"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_FILE = INPUT_DIR / "labels.csv"

# ----------------------------------
# LOAD METADATA
# ----------------------------------

df = pd.read_csv(CSV_FILE)

df = df[["filename", "diagnosis"]]

# ----------------------------------
# CREATE ARRAYS
# ----------------------------------

images = []
labels = []

processed = 0
failed = 0

# ----------------------------------
# PROCESS IMAGES
# ----------------------------------

for _, row in df.iterrows():

    image_path = INPUT_DIR / row["filename"]

    try:

        image = tf.io.read_file(str(image_path))

        #Gör om till pixlar med RBG
        image = tf.image.decode_jpeg(
            image,
            channels=3
        )
        
        #Resize
        image = tf.image.resize_with_pad(
            image,
            224,
            224
        )

        # Normalisering
        image = tf.cast(
            image,
            tf.float32
        ) / 255.0

        #Konverterar en bild från tensor till en numpy array,
        #Lägger till i en python lista
        images.append(image.numpy())
        labels.append(row["diagnosis"])

        processed += 1

        if processed % 500 == 0:
            print(f"Processed {processed} images")

    except Exception as e:

        failed += 1
        print(f"Failed: {image_path.name} -> {e}")

# ----------------------------------
# CONVERT TO NUMPY
# ----------------------------------

images = np.array(images, dtype=np.float32)
labels = np.array(labels)

# ----------------------------------
# SAVE DATA
# ----------------------------------

np.save(
    OUTPUT_DIR / "images.npy",
    images
)

np.save(
    OUTPUT_DIR / "labels.npy",
    labels
)

# ----------------------------------
# SUMMARY
# ----------------------------------

print("\nDone!")
print(f"Images processed: {processed}")
print(f"Errors: {failed}")

print("\nSaved files:")
print("images.npy")
print("labels.npy")

print("\nImage shape:")
print(images.shape)


Done!
Images processed: 20
Errors: 0

Saved files:
images.npy
labels.npy

Image shape:
(20, 224, 224, 3)
